In [2]:
import pandas as pd

df = pd.read_csv("tourism_weather_sun.csv")

In [5]:
eur = pd.read_csv("data/currency/euro.csv", sep=";")
usd = pd.read_csv("data/currency/usd.csv", sep=";")
nok = pd.read_csv("data/currency/dkk.csv", sep=";")
dkk = pd.read_csv("data/currency/nok.csv", sep=";")

In [6]:
eur.head()
eur.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 373 entries, 0 to 372
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Period  373 non-null    object
 1   Grupp   373 non-null    object
 2   Serie   373 non-null    object
 3   Medel   373 non-null    object
 4   Min     373 non-null    object
 5   Max     373 non-null    object
dtypes: object(6)
memory usage: 17.6+ KB


In [7]:
eur = eur[["Period", "Medel"]].copy()

eur = eur.rename(columns={
    "Period": "date",
    "Medel": "eur"
})

In [8]:
eur["eur"] = (
    eur["eur"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

In [9]:
eur["date"].head()

0      1995 april
1        1995 maj
2       1995 juni
3       1995 juli
4    1995 augusti
Name: date, dtype: object

In [10]:
month_map = {
    "januari": "01", "februari": "02", "mars": "03", "april": "04",
    "maj": "05", "juni": "06", "juli": "07", "augusti": "08",
    "september": "09", "oktober": "10", "november": "11", "december": "12"
}

eur["year"] = eur["date"].str.split().str[0]
eur["month_name"] = eur["date"].str.split().str[1]

eur["month_num"] = eur["month_name"].map(month_map)

eur["date"] = pd.to_datetime(
    eur["year"] + "-" + eur["month_num"] + "-01"
)

eur = eur[["date", "eur"]]

In [11]:
eur.head()
eur.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 373 entries, 0 to 372
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    373 non-null    datetime64[ns]
 1   eur     373 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 6.0 KB


In [12]:
eur.sample(10)

,date,eur
90,2002-10-01,9.10528
367,2025-11-01,10.99145
58,2000-02-01,8.51124
50,1999-06-01,8.83381
156,2008-04-01,9.37341
202,2012-02-01,8.82358
227,2014-03-01,8.86567
362,2025-06-01,11.00599
47,1999-03-01,8.94472
10,1996-02-01,8.62238


In [13]:
def load_currency_csv(file_path, currency_name):
    df = pd.read_csv(file_path, sep=";")

    df = df[["Period", "Medel"]].copy()
    df = df.rename(columns={
        "Period": "date",
        "Medel": currency_name
    })

    df[currency_name] = (
        df[currency_name]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

    month_map = {
        "januari": "01", "februari": "02", "mars": "03", "april": "04",
        "maj": "05", "juni": "06", "juli": "07", "augusti": "08",
        "september": "09", "oktober": "10", "november": "11", "december": "12"
    }

    df["year"] = df["date"].str.split().str[0]
    df["month_name"] = df["date"].str.split().str[1]
    df["month_num"] = df["month_name"].map(month_map)

    df["date"] = pd.to_datetime(
        df["year"] + "-" + df["month_num"] + "-01"
    )

    return df[["date", currency_name]]

In [14]:

usd = load_currency_csv("data/currency/usd.csv", "usd")
nok = load_currency_csv("data/currency/nok.csv", "nok")
dkk = load_currency_csv("data/currency/dkk.csv", "dkk")

In [15]:
currency_df = (
    eur
    .merge(usd, on="date", how="outer")
    .merge(nok, on="date", how="outer")
    .merge(dkk, on="date", how="outer")
    .sort_values("date")
    .reset_index(drop=True)
)

In [16]:
currency_df.head()
currency_df.tail()
currency_df.isna().sum()

date    0
eur     0
usd     0
nok     0
dkk     0
dtype: int64

In [17]:
currency_df.sample(20)

,date,eur,usd,nok,dkk
49,1999-05-01,8.97661,8.44276,1.08966,1.20763
94,2003-02-01,9.14988,8.49302,1.21305,1.23110
251,2016-03-01,9.28651,8.36898,0.98485,1.24535
140,2006-12-01,9.03771,6.83750,1.10758,1.21237
69,2001-01-01,8.89625,9.46693,1.08016,1.19186
180,2010-04-01,9.66985,7.20600,1.21792,1.29940
74,2001-06-01,9.20100,10.77526,1.15932,1.23434
100,2003-08-01,9.23495,8.28250,1.11733,1.24248
48,1999-04-01,8.91615,8.32438,1.07170,1.19958
125,2005-09-01,9.33666,7.62148,1.19655,1.25186


In [22]:
df["date"] = pd.to_datetime(df["date"])

In [23]:
df["month_date"] = (
    df["date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1028 entries, 0 to 1027
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   year                  1028 non-null   int64         
 1   guest_nights          1028 non-null   int64         
 2   region                1028 non-null   object        
 3   month                 1028 non-null   int64         
 4   date                  1028 non-null   datetime64[ns]
 5   occupancy_rate        1028 non-null   float64       
 6   guest_lag1            1028 non-null   float64       
 7   guest_lag2            1028 non-null   float64       
 8   guest_lag12           1028 non-null   float64       
 9   occ_lag1              1028 non-null   float64       
 10  temp_mean             1028 non-null   float64       
 11  rain_sum              1028 non-null   float64       
 12  rain_days             1028 non-null   float64       
 13  rain_mean         

In [25]:
currency_df["date"] = pd.to_datetime(currency_df["date"])

In [27]:

# byt namn på currency-datum så vi slipper date_x/date_y
currency_df = currency_df.rename(columns={"date": "month_date"})

# merge: alla dagar i samma månad får samma currency
df = df.merge(
    currency_df,
    on="month_date",
    how="left"
)

In [28]:
df[["date", "month_date", "eur", "usd", "nok", "dkk"]].head(10)

,date,month_date,eur,usd,nok,dkk
0,2016-07-01,2016-07-01,9.47149,8.56058,1.01087,1.27321
1,2016-07-01,2016-07-01,9.47149,8.56058,1.01087,1.27321
2,2016-07-01,2016-07-01,9.47149,8.56058,1.01087,1.27321
3,2016-07-01,2016-07-01,9.47149,8.56058,1.01087,1.27321
4,2016-07-01,2016-07-01,9.47149,8.56058,1.01087,1.27321
5,2016-07-01,2016-07-01,9.47149,8.56058,1.01087,1.27321
6,2016-07-01,2016-07-01,9.47149,8.56058,1.01087,1.27321
7,2016-07-01,2016-07-01,9.47149,8.56058,1.01087,1.27321
8,2016-07-01,2016-07-01,9.47149,8.56058,1.01087,1.27321
9,2016-08-01,2016-08-01,9.49340,8.47139,1.01956,1.27592


In [29]:
df = df.drop(columns=["month_date"], errors="ignore")

In [30]:
df.sample(20)

,year,guest_nights,region,month,date,occupancy_rate,guest_lag1,guest_lag2,guest_lag12,occ_lag1,...,rain_mean,guest_nights_total,occupancy_rate_total,share_of_total,occupancy_vs_total,sun_hours_day,eur,usd,nok,dkk
853,2024,1458091,stockholm,5,2024-05-01,0.705251,1111894.0,1103604.0,1420410.0,0.602557,...,0.738710,5598391.0,0.581083,0.260448,0.124168,14.702500,11.61375,10.73795,1.00185,1.55667
371,2019,461768,vastragotaland,12,2019-12-01,0.451905,575037.0,651573.0,448622.0,0.601698,...,4.393548,NaN,0.434972,NaN,0.016933,0.000000,10.49065,9.44239,1.04227,1.40396
496,2021,581058,dalarna,2,2021-02-01,0.448064,542154.0,317212.0,752758.0,0.354366,...,0.375000,2648072.0,0.275947,0.219427,0.172117,4.116944,10.08326,8.33380,0.98098,1.35584
816,2024,62760,vasternorrland,1,2024-01-01,0.389178,66408.0,76670.0,67274.0,0.384603,...,0.751613,3903626.0,0.425283,0.016077,-0.036105,0.325556,11.28338,10.34719,0.99412,1.51309
391,2020,134888,vasterbotten,2,2020-02-01,0.538906,96801.0,88799.0,124976.0,0.381843,...,2.779310,NaN,0.504657,NaN,0.034249,0.000000,10.57182,9.68850,1.04268,1.41497
158,2017,274114,skane,12,2017-12-01,0.438722,347918.0,378241.0,279420.0,0.594685,...,2.325806,NaN,0.456721,NaN,-0.017999,2.298611,9.94073,8.39863,1.00914,1.33555
462,2020,86642,vasternorrland,10,2020-10-01,0.499524,83003.0,148278.0,83819.0,0.489069,...,3.383871,3102050.0,0.400486,0.027931,0.099038,1.224757,10.39997,8.83982,0.95187,1.39743
194,2018,362836,dalarna,4,2018-04-01,0.391959,773900.0,766613.0,360134.0,0.610292,...,1.523333,NaN,0.539431,NaN,-0.147472,11.760556,10.37205,8.44590,1.07726,1.39262
339,2019,320948,norrbotten,8,2019-08-01,0.493550,631113.0,275151.0,309198.0,0.582407,...,4.161290,NaN,0.646539,NaN,-0.152988,14.395000,10.72882,9.64401,1.07574,1.43810
859,2024,229040,vasterbotten,6,2024-06-01,0.461268,132637.0,129169.0,196511.0,0.482061,...,1.723333,7260255.0,0.573851,0.031547,-0.112584,13.134722,11.28694,10.49285,0.98824,1.51315


In [41]:
import pandas as pd

df_final = pd.read_csv("final.csv")

In [42]:
df_final.sample(10)

,year,guest_nights,region,month,date,occupancy_rate,guest_lag1,guest_lag2,guest_lag12,occ_lag1,temp_mean,rain_sum,rain_days,rain_mean,occupancy_rate_total,occupancy_vs_total,sunny_days,mean_sun_hours,total_sun_hours
599,2022,274203,vastragotaland,1,2022-01-01,0.251193,432730.0,607694.0,178770.0,0.400792,1.6,59.7,14.0,1.925806,0.287972,-0.036779,7,2.232715,69.214167
558,2021,143651,jamtland,9,2021-09-01,0.459528,290237.0,440348.0,125842.0,0.520078,8.5,56.4,12.0,1.880000,0.493345,-0.033817,8,3.408546,102.256389
572,2021,147858,dalarna,10,2021-10-01,0.279953,206959.0,426166.0,164017.0,0.352989,8.0,85.7,12.0,2.764516,0.510355,-0.230402,6,2.565797,79.539722
664,2022,223256,gotland,8,2022-08-01,0.654820,360018.0,171970.0,260626.0,0.817862,19.1,46.3,7.0,1.493548,0.637468,0.017352,24,9.922222,307.588889
223,2018,2348954,vastragotaland,7,2018-07-01,0.792223,1163868.0,916987.0,2312030.0,0.684036,19.5,15.1,4.0,0.487097,0.674030,0.118193,26,11.010618,341.329167
907,2024,19292,gotland,11,2024-11-01,0.269684,39069.0,58199.0,18907.0,0.341062,5.6,18.2,3.0,0.606667,0.539874,-0.270191,3,2.003278,60.098333
214,2018,169048,vasterbotten,6,2018-06-01,0.481166,117940.0,126345.0,169818.0,0.494256,12.9,35.0,6.0,1.166667,0.602531,-0.121365,22,10.041583,301.247500
219,2018,738241,dalarna,7,2018-07-01,0.607094,293986.0,160444.0,737996.0,0.391543,20.4,42.9,5.0,1.383871,0.674030,-0.066936,26,12.063342,373.963611
303,2019,66979,vasternorrland,4,2019-04-01,0.455080,69977.0,67258.0,61664.0,0.455896,4.1,6.7,3.0,0.223333,0.499301,-0.044222,28,10.133068,303.992049
281,2019,492657,jamtland,2,2019-02-01,0.619409,325046.0,267595.0,492907.0,0.462304,-3.0,25.7,8.0,0.917857,0.511895,0.107514,3,2.354345,65.921667


In [43]:
# säkerställ datumformat
df_final["date"] = pd.to_datetime(df_final["date"])
df["date"] = pd.to_datetime(df["date"])

# plocka ut en ren currency-tabell från df
currency_cols = ["date", "eur", "usd", "nok", "dkk"]

currency_lookup = (
    df[currency_cols]
    .drop_duplicates(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)


# merge:a in valutorna i df_model
df_final = df_final.merge(
    currency_lookup,
    on="date",
    how="left"
)

In [44]:
df_final[["date", "eur", "usd", "nok", "dkk"]].head()

,date,eur,usd,nok,dkk
0,2016-07-01,9.47149,8.56058,1.01087,1.27321
1,2016-07-01,9.47149,8.56058,1.01087,1.27321
2,2016-07-01,9.47149,8.56058,1.01087,1.27321
3,2016-07-01,9.47149,8.56058,1.01087,1.27321
4,2016-07-01,9.47149,8.56058,1.01087,1.27321


In [45]:
df_final[["eur", "usd", "nok", "dkk"]].isna().sum()

eur    0
usd    0
nok    0
dkk    0
dtype: int64

In [46]:
df_final.sample(20)

,year,guest_nights,region,month,date,occupancy_rate,guest_lag1,guest_lag2,guest_lag12,occ_lag1,...,rain_mean,occupancy_rate_total,occupancy_vs_total,sunny_days,mean_sun_hours,total_sun_hours,eur,usd,nok,dkk
89,2017,608760,vastragotaland,4,2017-04-01,0.572254,474600.0,432032.0,534857.0,0.559655,...,1.656667,0.510602,0.061653,15,6.849120,205.473611,9.59029,8.94395,1.04165,1.28943
919,2025,281575,skane,1,2025-01-01,0.370032,340254.0,375271.0,260309.0,0.448682,...,1.693548,0.410187,-0.040154,3,1.103369,34.204444,11.48040,11.09257,0.97729,1.53872
827,2024,320834,skane,2,2024-02-01,0.464048,260309.0,315449.0,328570.0,0.394293,...,2.903448,0.489223,-0.025175,2,1.630661,47.289167,11.24998,10.42206,0.98822,1.50905
616,2022,83218,vasternorrland,3,2022-03-01,0.509388,63933.0,52317.0,47232.0,0.400098,...,0.135484,0.490135,0.019253,22,7.504364,232.635278,10.55140,9.57152,1.08345,1.41811
844,2024,129169,vasterbotten,4,2024-04-01,0.454516,136632.0,136871.0,129191.0,0.439027,...,1.410000,0.506603,-0.052087,17,7.106472,213.194167,11.59096,10.80559,0.99214,1.55384
970,2025,2561995,vastragotaland,7,2025-07-01,0.787205,1259865.0,1038269.0,2489238.0,0.591096,...,2.509677,0.674208,0.112997,22,8.193943,254.012222,11.19852,9.59089,0.94473,1.50064
36,2016,104872,jamtland,11,2016-11-01,0.403084,83352.0,106875.0,91683.0,0.361519,...,1.823333,0.575400,-0.172316,0,0.951546,28.546389,9.85036,9.11263,1.08446,1.32384
604,2022,378070,vastragotaland,2,2022-02-01,0.369356,274203.0,432730.0,219448.0,0.251193,...,4.785714,0.394250,-0.024894,7,3.185476,89.193333,10.54224,9.29104,1.04775,1.41586
340,2019,915541,skane,8,2019-08-01,0.710282,1361830.0,748467.0,853272.0,0.792022,...,1.535484,0.646539,0.063743,22,7.636237,236.723333,10.72882,9.64401,1.07574,1.43810
917,2024,436148,dalarna,12,2024-12-01,0.398243,132831.0,137787.0,404335.0,0.337783,...,2.641935,0.429811,-0.031568,0,1.544964,47.893889,11.50489,10.96938,0.98035,1.54244


In [47]:
df_final.to_csv(
    "data/final_tourist_data.csv",
    index=False
)